In [2]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import fdrcorrection
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from mrmr import mrmr_classif

# ==========================================
# 1. DATA IMPORT & PREPARATION
# ==========================================
print("Loading and preparing dataset...")
raw_df = pd.read_csv("golub.csv", index_col=0).copy()
label_map = {"allB": 0, "allT": 0, "aml": 1}
raw_df["target"] = raw_df["cancer"].map(label_map)

metadata_cols = ["Samples", "BM.PB", "Gender", "Source", "tissue.mf", "cancer", "target"]
feature_cols = [column for column in raw_df.columns if column not in metadata_cols]

X = raw_df[feature_cols]
y = raw_df["target"]

# Convert to NumPy arrays for faster cross-validation slicing
X_array = X.values
y_array = y.values

# ==========================================
# 2. SETUP PIPELINE PARAMETERS
# ==========================================
MRMR_FEATURES = 100        # Genes to keep after redundancy filtering
FINAL_PANEL_SIZE = 12      # Final biomarkers to keep after SVM-RFE
N_SPLITS = 5               # Number of cross-validation folds

outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_metrics = []
all_selected_biomarkers = []

print(f"\nStarting {N_SPLITS}-Fold Cross-Validation for Hybrid mRMR + SVM-RFE...\n")

# ==========================================
# 3. EXPERIMENTATION LOOP
# ==========================================
for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_array, y_array)):
    print(f"--- Processing Fold {fold + 1}/{N_SPLITS} ---")
    
    # 3a. Split Data
    X_train, X_test = X_array[train_idx], X_array[test_idx]
    y_train, y_test = y_array[train_idx], y_array[test_idx]
    
    # 3b. Scale Data (Fit on Train, Transform on Test to prevent data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ------------------------------------------
    # PHASE 2: STATISTICAL BASELINE (Training Data Only)
    # ------------------------------------------
    X_train_ALL = X_train_scaled[y_train == 0]
    X_train_AML = X_train_scaled[y_train == 1]
    
    # T-test
    _, p_vals_t = ttest_ind(X_train_ALL, X_train_AML, equal_var=False, axis=0)
    _, p_adj_t = fdrcorrection(np.nan_to_num(p_vals_t, nan=1.0), alpha=0.01)
    sig_t = set(np.array(feature_cols)[p_adj_t < 0.01])
    
    # Mann-Whitney U test
    p_vals_mw = [mannwhitneyu(X_train_ALL[:, i], X_train_AML[:, i]).pvalue for i in range(X_train_scaled.shape[1])]
    _, p_adj_mw = fdrcorrection(p_vals_mw, alpha=0.01)
    sig_mw = set(np.array(feature_cols)[p_adj_mw < 0.01])
    
    robust_baseline_genes = sig_t.intersection(sig_mw)

    # ------------------------------------------
    # STAGE 1: mRMR PRE-FILTERING
    # ------------------------------------------
    # mRMR requires pandas DataFrames
    X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
    y_train_series = pd.Series(y_train)
    
    # Silence mRMR progress bars by routing to devnull or just let it print
    selected_mrmr_genes = mrmr_classif(X=X_train_df, y=y_train_series, K=MRMR_FEATURES, show_progress=False)
    mrmr_indices = [feature_cols.index(gene) for gene in selected_mrmr_genes]
    
    # Subset to only mRMR approved features
    X_train_mrmr = X_train_scaled[:, mrmr_indices]
    X_test_mrmr = X_test_scaled[:, mrmr_indices]

    # ------------------------------------------
    # STAGE 2: SVM-RFE
    # ------------------------------------------
    svm_estimator = SVC(kernel="linear", C=1.0, random_state=42)
    svm_rfe = RFE(estimator=svm_estimator, n_features_to_select=FINAL_PANEL_SIZE, step=1)
    
    # Fit the RFE model on the mRMR-filtered data
    svm_rfe.fit(X_train_mrmr, y_train)
    
    # Extract the final genes
    rfe_support = svm_rfe.support_
    final_hybrid_genes = [selected_mrmr_genes[i] for i, mask in enumerate(rfe_support) if mask]
    all_selected_biomarkers.extend(final_hybrid_genes)

    # ------------------------------------------
    # EVALUATION ON HOLD-OUT TEST SET
    # ------------------------------------------
    y_pred = svm_rfe.predict(X_test_mrmr)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Calculate Concordance against the fold's strict statistical baseline
    overlap = set(final_hybrid_genes).intersection(robust_baseline_genes)
    concordance = len(overlap) / len(final_hybrid_genes) if len(final_hybrid_genes) > 0 else 0
    
    fold_metrics.append({
        'Fold': fold + 1,
        'Stable_Genes': len(final_hybrid_genes),
        'Concordance': concordance,
        'Accuracy': acc,
        'F1_Score': f1
    })
    
    print(f"  -> Concordance: {concordance*100:.1f}% | Accuracy: {acc*100:.1f}%\n")

# ==========================================
# 4. FINAL SYNTHESIS & REPORTING
# ==========================================
results_df = pd.DataFrame(fold_metrics)

print("==========================================")
print(" FINAL mRMR + SVM-RFE CV RESULTS")
print("==========================================")
print(results_df.to_string(index=False))

print("\n--- Averages Across All 5 Folds ---")
print(f"Average Concordance Rate:      {results_df['Concordance'].mean()*100:.1f}%")
print(f"Average Test Accuracy:         {results_df['Accuracy'].mean()*100:.1f}%")
print(f"Average Test F1-Score:         {results_df['F1_Score'].mean():.4f}")

# Identify the "Super-Biomarkers" (Genes selected in multiple folds)
global_gene_counts = Counter(all_selected_biomarkers)
print("\n--- Global 'Super-Biomarkers' (Selected in >= 3 Folds) ---")
for gene, count in global_gene_counts.most_common():
    if count >= 3:
        print(f"{gene}: Selected in {count}/5 Folds")

Loading and preparing dataset...

Starting 5-Fold Cross-Validation for Hybrid mRMR + SVM-RFE...

--- Processing Fold 1/5 ---
  -> Concordance: 75.0% | Accuracy: 100.0%

--- Processing Fold 2/5 ---
  -> Concordance: 100.0% | Accuracy: 100.0%

--- Processing Fold 3/5 ---
  -> Concordance: 91.7% | Accuracy: 100.0%

--- Processing Fold 4/5 ---
  -> Concordance: 100.0% | Accuracy: 85.7%

--- Processing Fold 5/5 ---
  -> Concordance: 83.3% | Accuracy: 100.0%

 FINAL mRMR + SVM-RFE CV RESULTS
 Fold  Stable_Genes  Concordance  Accuracy  F1_Score
    1            12     0.750000  1.000000       1.0
    2            12     1.000000  1.000000       1.0
    3            12     0.916667  1.000000       1.0
    4            12     1.000000  0.857143       0.8
    5            12     0.833333  1.000000       1.0

--- Averages Across All 5 Folds ---
Average Concordance Rate:      90.0%
Average Test Accuracy:         97.1%
Average Test F1-Score:         0.9600

--- Global 'Super-Biomarkers' (Selected i